In [0]:
from pyspark.sql import SparkSession

In [0]:
spark

TURN OFF AQE

In [0]:
# Note: spark.sql.adaptive.enabled cannot be modified on Serverless compute
# Adaptive Query Execution (AQE) is managed server-side and is enabled by default
print("Adaptive Query Execution (AQE) is managed by the platform on Serverless compute")

In [0]:
from pyspark.sql.functions import *

In [0]:
# the cell shows df reading the data from DBFS, as the DBFS is disabled in databricks free account hence we'll not be able to use the data from DBFS we can read teh table from unity catalog

df=spark.read.format('CSV').option('inferschema','True').option('header','True').load('/dbacademy/default/big_mart_sales')



#Reading data from unity catalog table 

**Method 1:** Using spark.table() (Recommended)

`df = spark.table('catalog.schema.table_name')
`

**Method 2:** Using spark.read.table()

`df = spark.read.table('catalog.schema.table_name')`

**Method 3:** Using SQL

`df = spark.sql("SELECT * FROM catalog.schema.table_name")`




For now I'll read the file from different location

In [0]:
df=spark.read.format('CSV').option('inferschema','True').option('header','True').load('/Workspace/Shared/BigMart Sales.csv')
df.display()

###Get Number of Partitions

In [0]:
df.rdd.getNumPartitions()


#[NOT_IMPLEMENTED] Using custom code using PySpark RDDs is not allowed on serverless compute. We suggest using mapInPandas or mapInArrow for the most common use cases, or switch to Dedicated access mode if you require RDDs. For more details on compatibility and limitations, check: https://docs.databricks.com/release-notes/serverless.html#limitations


In [0]:

# trying to get the patition info using the below method :

# def count_partitions(iterator):
#     yield 1

# num_partitions = df.mapInPandas(count_partitions, "int").count()
# num_partitions

In [0]:
df.select(spark_partition_id()).distinct().count()

# output: 1

###Change defalut Partition Size 

In [0]:
spark.conf.set('spark.sql.files.maxPartitionsPerFile','131072')


#[CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.sql.files.maxPartitionsPerFile is not available.  SQLSTATE: 42K0I



##Repartition

`df=df.repartition(n)`

this will divide the data in the dataframe df into 'n' number of Logical partitions of **EQUAL SIZE**

In [0]:
df=df.repartition(10)

In [0]:
# Get number of partitions using Spark Connect-compatible method
df.select(spark_partition_id()).distinct().count()


#output: 10 

### Get the partition info

we use **spark_partition_id()** function to get the id of partition where that data is stored.

In [0]:
df.withColumn('partition_id',spark_partition_id()).display()

### writing the data to see partitions

In [0]:
df.write.format('parquet').mode('overwrite').option('path','/Workspace/Shared/Writing partition data').save()

# this writes the data at the provided path into 10 partitions since we used repartition on the df and created 10 partitions

###new data reading 

In [0]:
df_new=spark.read.format('parquet').load('/Workspace/Shared/Writing partition data')

In [0]:
df_new.display()

# we can check the number of files spark reads when we display the data frame that we created by reading the parquet files
# we can see that spark reads 10 files, which is the number of partitions we created


In [0]:
df_new.filter(col('Outlet_Location_Type')=='Tier 1').display()

# we can see it reads all the 10 files since we don't know the partitioning where the information for the pruning  condition is stored


### Scanning data optimization

In [0]:
df_new.write.format('parquet')\
    .mode('append')\
        .partitionBy('Outlet_Location_Type')\
            .option('path','/Workspace/Shared/Writing partition data_opt')\
                .save()

In [0]:
df_new=spark.read.format('parquet').load('/Workspace/Shared/Writing partition data_opt')

In [0]:
df_new.display()

#now it shows it is reading 15 files

In [0]:
df_new=df_new.filter(col('Outlet_Location_Type')=='Tier 1')
df_new.display()

#now it only reads the 5 files that is the number of files present in the Outlet_Location_Type=Tier 1 folder 

## Broadcast Join

In [0]:
# Big DataFrame
df_transactions = spark.createDataFrame([
    (1, "US", 100),
    (2, "IN", 200),
    (3, "UK", 150),
    (4, "US", 80),
], ["id", "country_code", "amount"])

# Small DataFrame
df_countries = spark.createDataFrame([
    ("US", "United States"),
    ("IN", "India"),
    ("UK", "United Kingdom"),
], ["country_code", "country_name"])

In [0]:
df_transactions.display()

In [0]:
df_countries.display()

In [0]:
df_transactions.join(df_countries, on=["country_code"],how='inner').display()


#Broadcast join 





In [0]:
df_transactions.join(broadcast(df_countries), on=["country_code"],how='inner').display()

#broadcast is used to broadcast the small dataframe to all the nodes
#it is used to reduce the network traffic
#it is used to reduce the shuffle
#it is used to reduce the time taken to join the dataframes
#it is used to reduce the memory usage

#'Broadcast is not supported in Spark Connect.databricks-flint(SCPCCBA010E0FF0CB)'
